# Klasifikasi Bunga — VGG16 Fine-Tuning
## Mata Kuliah Machine Learning — UAS

**Dataset:** Bunga Melati Jakarta, Melati Jepang, Bintaro, dan Tapak Dara  
**Metode:** VGG16 Transfer Learning + Fine-Tuning (End-to-End)  
**Referensi:** Fei et al. (2023) — *A Lightweight Attention-Based Convolutional Neural Networks for Fresh-Cut Flower Classification*, IEEE Access 11, 17283–17293

---
### Alur Penelitian:
1. Import Library
2. Konfigurasi & Struktur Dataset
3. Load & Preprocessing Dataset
4. Data Augmentation
5. Arsitektur VGG16 Fine-Tuning
6. Training — Phase 1 (Head Training)
7. Training — Phase 2 (Fine-Tuning Conv5)
8. Evaluasi Model
9. Visualisasi Hasil
10. Demo Prediksi & Simpan Model

## 1. Import Library

In [ ]:
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    matthews_corrcoef
)

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

print(f'TensorFlow  : {tf.__version__}')
print(f'GPU         : {tf.config.list_physical_devices("GPU")}')
print('Semua library berhasil diimpor!')

## 2. Konfigurasi

Struktur folder dataset yang dibutuhkan:
```
dataset/
├── melati_jakarta/     (360 gambar)
├── melati_jepang/      (360 gambar)
├── bintaro/            (360 gambar)
└── tapak_dara/         (360 gambar)
```

In [ ]:
# ============================================================
# KONFIGURASI PARAMETER
# ============================================================

# ⚠️ SESUAIKAN PATH DATASET ANDA
DATASET_PATH  = './dataset'

# Kelas
CLASS_NAMES   = ['melati_jakarta', 'melati_jepang', 'bintaro', 'tapak_dara']
CLASS_LABELS  = ['Melati Jakarta', 'Melati Jepang', 'Bintaro', 'Tapak Dara']
NUM_CLASSES   = len(CLASS_NAMES)
COLORS        = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

# Gambar
IMG_SIZE      = 224        # Ukuran standar input VGG16

# Training
BATCH_SIZE    = 16         # Disesuaikan untuk dataset ~1.440 gambar
EPOCHS_P1     = 15         # Phase 1: head training
EPOCHS_P2     = 10         # Phase 2: fine-tuning conv5
LR            = 0.001      # Learning rate awal
TEST_SIZE     = 0.15       # 15% test  — ketentuan UAS
VAL_SIZE      = 0.15       # 15% val   — ketentuan UAS
RANDOM_STATE  = 42

print('Konfigurasi berhasil disimpan.')
for k, v in {
    'Kelas'         : CLASS_NAMES,
    'Input VGG16'   : f'{IMG_SIZE}x{IMG_SIZE}x3',
    'Batch size'    : BATCH_SIZE,
    'Epochs P1/P2'  : f'{EPOCHS_P1}/{EPOCHS_P2}',
    'Split'         : f'Train 70% | Val {int(VAL_SIZE*100)}% | Test {int(TEST_SIZE*100)}%'
}.items():
    print(f'  {k:<16}: {v}')

## 3. Load & Preprocessing Dataset

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

def load_dataset(path, class_names, img_size):
    """
    Memuat seluruh gambar dari folder dataset.
    Setiap subfolder = satu kelas.

    Return:
        X : np.array gambar RGB (N, img_size, img_size, 3)
        y : np.array label integer (N,)
    """
    X, y = [], []
    exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif')

    for label, name in enumerate(class_names):
        folder = os.path.join(path, name)
        if not os.path.exists(folder):
            print(f'  ⚠️  Folder tidak ditemukan: {folder}')
            continue
        files = [f for f in os.listdir(folder) if f.lower().endswith(exts)]
        print(f'  [{name:<18}] {len(files):>4d} gambar')
        for f in files:
            try:
                img = Image.open(os.path.join(folder, f)).convert('RGB')
                img = img.resize((img_size, img_size))
                X.append(np.array(img, dtype=np.uint8))
                y.append(label)
            except Exception as e:
                print(f'    ⚠️  Skip {f}: {e}')

    return np.array(X), np.array(y)


print('Loading dataset...')
X_raw, y = load_dataset(DATASET_PATH, CLASS_NAMES, IMG_SIZE)

print(f'\nTotal  : {len(X_raw)} gambar | Shape: {X_raw.shape}')
for i, lbl in enumerate(CLASS_LABELS):
    print(f'  {lbl:<18}: {np.sum(y==i)} gambar')

In [ ]:
# ============================================================
# VISUALISASI DISTRIBUSI DATASET
# ============================================================

unique, counts = np.unique(y, return_counts=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribusi Dataset Bunga', fontsize=13, fontweight='bold')

bars = axes[0].bar(CLASS_LABELS, counts, color=COLORS, edgecolor='black')
axes[0].set_title('Jumlah Gambar per Kelas')
axes[0].set_xlabel('Kelas Bunga')
axes[0].set_ylabel('Jumlah Gambar')
axes[0].tick_params(axis='x', rotation=15)
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 2, str(cnt),
                 ha='center', fontweight='bold')

axes[1].pie(counts, labels=CLASS_LABELS, autopct='%1.1f%%',
            colors=COLORS, startangle=90, pctdistance=0.82)
axes[1].set_title('Proporsi per Kelas')

plt.tight_layout()
plt.savefig('distribusi_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: distribusi_dataset.png')

In [ ]:
# ============================================================
# VISUALISASI SAMPEL DATASET
# ============================================================

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle('Sampel Dataset Bunga (5 gambar per kelas)', fontsize=14, fontweight='bold')

for i in range(NUM_CLASSES):
    idx_c = np.where(y == i)[0]
    samps = np.random.choice(idx_c, min(5, len(idx_c)), replace=False)
    for j, idx in enumerate(samps):
        axes[i, j].imshow(X_raw[idx])
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(CLASS_LABELS[i], fontsize=10,
                                  fontweight='bold', color=COLORS[i])

plt.tight_layout()
plt.savefig('sampel_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: sampel_dataset.png')

In [ ]:
# ============================================================
# PREPROCESSING
# VGG16 menggunakan preprocess_input (zero-center per
# ImageNet channel mean), bukan normalisasi [0,1] biasa
# ============================================================

X_vgg = preprocess_input(X_raw.astype(np.float32))  # shape: (N, 224, 224, 3)
y_oh  = to_categorical(y, NUM_CLASSES)               # One-hot untuk training

print(f'X_vgg shape : {X_vgg.shape}   ← input VGG16')
print(f'y_oh shape  : {y_oh.shape}    ← one-hot label')

In [ ]:
# ============================================================
# SPLIT DATA — Train 70% / Val 15% / Test 15%
# Sesuai ketentuan UAS
# ============================================================

val_ratio = VAL_SIZE / (1 - TEST_SIZE)  # proporsi val dari sisa setelah test

# Tahap 1: pisahkan test (15%)
X_tv, X_test_v, y_tv_oh, y_test_oh, y_tv, y_test = train_test_split(
    X_vgg, y_oh, y,
    test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# Tahap 2: pisahkan val (15%) dari sisa 85%
X_train_v, X_val_v, y_train_oh, y_val_oh, y_train_v, y_val_v = train_test_split(
    X_tv, y_tv_oh, y_tv,
    test_size=val_ratio, stratify=y_tv, random_state=RANDOM_STATE
)

n = len(X_vgg)
print('Pembagian Dataset:')
for name, size in [('Train', len(X_train_v)), ('Validasi', len(X_val_v)), ('Test', len(X_test_v))]:
    print(f'  {name:<10}: {size:>4d} gambar ({size/n*100:.1f}%)')

## 4. Data Augmentation

Mengacu pada Fei et al. (2023) Section III-A-2: augmentasi meliputi horizontal flip, rotasi, zoom, shift, dan perubahan kecerahan. Augmentasi **hanya** diterapkan pada data training, tidak pada validasi dan test.

In [ ]:
# ============================================================
# DATA AUGMENTATION
# Referensi: Fei et al. (2023) Figure 5 — teknik augmentasi
# ============================================================

train_aug = ImageDataGenerator(
    horizontal_flip    = True,         # Flip horizontal
    rotation_range     = 90,           # Rotasi hingga 90°
    zoom_range         = 0.20,         # Zoom in/out
    width_shift_range  = 0.15,         # Geser horizontal
    height_shift_range = 0.15,         # Geser vertikal
    brightness_range   = [0.7, 1.3],   # Variasi kecerahan
    shear_range        = 0.10,         # Shear transform
    fill_mode          = 'nearest'
)
no_aug = ImageDataGenerator()  # Validasi: tanpa augmentasi

train_gen = train_aug.flow(X_train_v, y_train_oh, batch_size=BATCH_SIZE, shuffle=True)
val_gen   = no_aug.flow(X_val_v,   y_val_oh,   batch_size=BATCH_SIZE, shuffle=False)

steps_ep  = max(1, len(X_train_v) // BATCH_SIZE)
val_steps = max(1, len(X_val_v)   // BATCH_SIZE)

print(f'Steps per epoch : {steps_ep}')
print(f'Val steps       : {val_steps}')

# Visualisasi contoh augmentasi
sample  = X_raw[0]
aug_vis = train_aug.flow(sample[np.newaxis].astype(np.float32), batch_size=1)
titles  = ['Original', 'H-Flip', 'Rotasi', 'Zoom',
           'Shift', 'Brightness+', 'Brightness-', 'Shear', 'Combined']

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Teknik Data Augmentation (Fei et al., 2023)',
             fontsize=13, fontweight='bold')
axes[0, 0].imshow(sample)
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')
for k in range(1, 10):
    aug_img = next(aug_vis)[0]
    aug_img = np.clip(aug_img + 123, 0, 255).astype(np.uint8)
    r, c = k // 5, k % 5
    axes[r, c].imshow(aug_img)
    axes[r, c].set_title(titles[k] if k < len(titles) else f'Aug {k}', fontsize=9)
    axes[r, c].axis('off')

plt.tight_layout()
plt.savefig('data_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: data_augmentation.png')

## 5. Arsitektur Model VGG16

Mengacu pada Fei et al. (2023): *"The common method is to use VGG, Inception, ResNet, and other classic architectures for transfer learning."*

```
Input (224×224×3)
  → VGG16 Backbone (pre-trained ImageNet, frozen di Phase 1)
  → GlobalAveragePooling2D  → 512-D
  → Dense(1024, ReLU) + BatchNorm + Dropout(0.5)
  → Dense(256, ReLU)  + Dropout(0.3)
  → Dense(4, Softmax)
```

In [ ]:
# ============================================================
# BANGUN ARSITEKTUR VGG16
# ============================================================

print('=' * 58)
print('  ARSITEKTUR VGG16 FINE-TUNING')
print('=' * 58)

# Load VGG16 tanpa top layer, pre-trained ImageNet
base_vgg = VGG16(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (IMG_SIZE, IMG_SIZE, 3)
)
base_vgg.trainable = False   # Freeze untuk Phase 1

# Custom classification head
x   = base_vgg.output
x   = GlobalAveragePooling2D(name='gap')(x)           # 512-D
x   = Dense(1024, activation='relu', name='fc1')(x)
x   = BatchNormalization(name='bn1')(x)
x   = Dropout(0.5, name='drop1')(x)
x   = Dense(256, activation='relu', name='fc2')(x)
x   = Dropout(0.3, name='drop2')(x)
out = Dense(NUM_CLASSES, activation='softmax', name='softmax')(x)

vgg_e2e = Model(inputs=base_vgg.input, outputs=out, name='VGG16_FineTuning')

# Ringkasan parameter
total     = vgg_e2e.count_params()
trainable = sum([tf.size(w).numpy() for w in vgg_e2e.trainable_weights])
print(f'Total parameter     : {total:,}')
print(f'Parameter trainable : {trainable:,}  (hanya head)')
print(f'Parameter frozen    : {total - trainable:,}  (VGG16 backbone)')

## 6. Training — Phase 1 (Head Training)

Backbone VGG16 **dibekukan**, hanya melatih classification head baru.

In [ ]:
# ============================================================
# PHASE 1 — HEAD TRAINING (backbone frozen)
# ============================================================

vgg_e2e.compile(
    optimizer = Adam(LR),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

cb_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_vgg16_phase1.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0)
]

print(f'Phase 1: Training classification head (backbone frozen, max {EPOCHS_P1} epoch)...')
hist_p1 = vgg_e2e.fit(
    train_gen,
    steps_per_epoch  = steps_ep,
    epochs           = EPOCHS_P1,
    validation_data  = val_gen,
    validation_steps = val_steps,
    callbacks        = cb_p1,
    verbose          = 1
)
print('Phase 1 selesai!')

## 7. Training — Phase 2 (Fine-Tuning Conv5)

Buka 4 layer terakhir backbone (blok conv5) untuk dilatih ulang dengan learning rate yang lebih kecil.

In [ ]:
# ============================================================
# PHASE 2 — FINE-TUNING BLOK CONV5 VGG16
# ============================================================

# Buka backbone, freeze semua kecuali 4 layer terakhir
base_vgg.trainable = True
for layer in base_vgg.layers[:-4]:
    layer.trainable = False

# Compile ulang dengan LR lebih kecil (1/10 dari Phase 1)
vgg_e2e.compile(
    optimizer = Adam(LR / 10),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

cb_p2 = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint('best_vgg16_finetuned.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0)
]

# Parameter trainable setelah dibuka
trainable_p2 = sum([tf.size(w).numpy() for w in vgg_e2e.trainable_weights])
print(f'Parameter trainable Phase 2: {trainable_p2:,}')
print(f'Phase 2: Fine-tuning blok conv5 (max {EPOCHS_P2} epoch)...')

hist_p2 = vgg_e2e.fit(
    train_gen,
    steps_per_epoch  = steps_ep,
    epochs           = EPOCHS_P2,
    validation_data  = val_gen,
    validation_steps = val_steps,
    callbacks        = cb_p2,
    verbose          = 1
)
print('Training VGG16 selesai!')

In [ ]:
# ============================================================
# GRAFIK TRAINING — Accuracy & Loss (Phase 1 + Phase 2)
# ============================================================

def merge_hist(h1, h2):
    """Gabungkan history Phase 1 dan Phase 2."""
    return {k: h1.history[k] + h2.history.get(k, []) for k in h1.history}

hist_all = merge_hist(hist_p1, hist_p2)
ep_split = len(hist_p1.history['loss'])   # titik mulai fine-tuning
epochs_r = range(1, len(hist_all['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Grafik Training — VGG16 Fine-Tuning', fontsize=13, fontweight='bold')

for ax, (tk, vk), ylabel in zip(
    axes,
    [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
    ['Accuracy', 'Loss']
):
    ax.plot(epochs_r, hist_all[tk],  label='Training',  color='#2196F3', lw=2)
    ax.plot(epochs_r, hist_all[vk],  label='Validasi',  color='#FF9800', ls='--', lw=2)
    ax.axvline(ep_split, color='red', ls=':', lw=1.5, alpha=0.7,
               label=f'Fine-tuning mulai (epoch {ep_split+1})')
    ax.set_title(f'Grafik {ylabel}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
    if ylabel == 'Accuracy':
        ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('grafik_training_vgg16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: grafik_training_vgg16.png')

## 8. Evaluasi Model

In [ ]:
# ============================================================
# FUNGSI EVALUASI
# Metrik: Accuracy, Precision, Recall, F1-Score,
#         Specificity, MCC, CV 5-Fold
# ============================================================

from sklearn.model_selection import cross_val_score

def compute_metrics(y_true, y_pred, name):
    """
    Menghitung dan menampilkan semua metrik evaluasi.
    Menggunakan average='macro' untuk evaluasi berimbang antar kelas.
    """
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)

    # Specificity rata-rata per kelas
    cm = confusion_matrix(y_true, y_pred)
    spec_list = []
    for i in range(NUM_CLASSES):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        spec_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    spec = np.mean(spec_list)

    print(f'\n  {"─"*45}')
    print(f'  HASIL EVALUASI — {name}')
    print(f'  {"─"*45}')
    print(f'  Accuracy     : {acc*100:.2f}%')
    print(f'  Precision    : {prec:.4f}')
    print(f'  Recall       : {rec:.4f}')
    print(f'  Specificity  : {spec:.4f}')
    print(f'  F1-Score     : {f1:.4f}')
    print(f'  MCC          : {mcc:.4f}')
    print(f'  {"─"*45}')

    return {
        'Model'       : name,
        'Accuracy (%)': round(acc * 100, 2),
        'Precision'   : round(prec, 4),
        'Recall'      : round(rec, 4),
        'Specificity' : round(spec, 4),
        'F1-Score'    : round(f1, 4),
        'MCC'         : round(mcc, 4)
    }


# Prediksi pada data test
print('Memprediksi data test...')
y_prob_vgg = vgg_e2e.predict(X_test_v, batch_size=BATCH_SIZE, verbose=0)
y_pred_vgg = np.argmax(y_prob_vgg, axis=1)

vgg_metrics = compute_metrics(y_test, y_pred_vgg, 'VGG16 Fine-Tuning')

print('\nClassification Report — VGG16:')
print(classification_report(y_test, y_pred_vgg, target_names=CLASS_LABELS, zero_division=0))

## 9. Visualisasi Hasil

In [ ]:
# ============================================================
# CONFUSION MATRIX — Raw + Normalisasi
# ============================================================

cm_vgg      = confusion_matrix(y_test, y_pred_vgg)
cm_vgg_norm = cm_vgg.astype('float') / cm_vgg.sum(axis=1)[:, np.newaxis]
short_lbl   = [c.replace(' ', '\n') for c in CLASS_LABELS]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Confusion Matrix — VGG16 Fine-Tuning\nAkurasi: {vgg_metrics["Accuracy (%)"]:.2f}%',
             fontsize=13, fontweight='bold')

sns.heatmap(cm_vgg, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_lbl, yticklabels=short_lbl,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Jumlah Prediksi', fontweight='bold')
axes[0].set_ylabel('Aktual')
axes[0].set_xlabel('Prediksi')

sns.heatmap(cm_vgg_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=short_lbl, yticklabels=short_lbl,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Normalisasi', fontweight='bold')
axes[1].set_ylabel('Aktual')
axes[1].set_xlabel('Prediksi')

plt.tight_layout()
plt.savefig('confusion_matrix_vgg16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: confusion_matrix_vgg16.png')

In [ ]:
# ============================================================
# BAR CHART METRIK PER KELAS
# ============================================================

report_dict = classification_report(
    y_test, y_pred_vgg,
    target_names=CLASS_LABELS,
    output_dict=True,
    zero_division=0
)
metrics_df = pd.DataFrame({
    cls: {
        'Precision': report_dict[cls]['precision'],
        'Recall'   : report_dict[cls]['recall'],
        'F1-Score' : report_dict[cls]['f1-score']
    }
    for cls in CLASS_LABELS
}).T

ax = metrics_df.plot(
    kind='bar', figsize=(11, 5),
    color=['#1565C0', '#2E7D32', '#E65100'],
    edgecolor='black', width=0.7
)
plt.title('Metrik Evaluasi per Kelas — VGG16', fontsize=12, fontweight='bold')
plt.xlabel('Kelas Bunga')
plt.ylabel('Nilai Metrik')
plt.xticks(rotation=15)
plt.ylim(0, 1.15)
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=2, fontsize=9)
plt.tight_layout()
plt.savefig('metrik_per_kelas_vgg16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: metrik_per_kelas_vgg16.png')

In [ ]:
# ============================================================
# RINGKASAN METRIK AKHIR (Bar Chart)
# ============================================================

metric_names  = ['Accuracy (%)', 'Precision', 'Recall', 'F1-Score', 'MCC']
metric_values = [
    vgg_metrics['Accuracy (%)'],
    vgg_metrics['Precision'] * 100,
    vgg_metrics['Recall'] * 100,
    vgg_metrics['F1-Score'] * 100,
    vgg_metrics['MCC'] * 100
]
bar_colors = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A', '#00838F']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(metric_names, metric_values, color=bar_colors,
              edgecolor='black', width=0.5)
ax.set_title('Ringkasan Metrik Evaluasi — VGG16 Fine-Tuning (Test Set)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Nilai (%)')
ax.set_ylim(0, 118)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.2f}%', ha='center', va='bottom',
            fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('ringkasan_metrik_vgg16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: ringkasan_metrik_vgg16.png')

## 10. Demo Prediksi & Simpan Model

In [ ]:
# ============================================================
# DEMO PREDIKSI — 8 Sampel dari Data Test
# ============================================================

np.random.seed(RANDOM_STATE)
demo_idx  = np.random.choice(len(X_test_v), 8, replace=False)
X_demo    = X_test_v[demo_idx]
prob_demo = vgg_e2e.predict(X_demo, verbose=0)
pred_demo = np.argmax(prob_demo, axis=1)
true_demo = y_test[demo_idx]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Demo Prediksi — VGG16 Fine-Tuning', fontsize=13, fontweight='bold')

for k, ax in enumerate(axes.flat):
    img_show = np.clip(X_demo[k] + 123, 0, 255).astype(np.uint8)
    ax.imshow(img_show)

    true_lbl = CLASS_LABELS[true_demo[k]]
    pred_lbl = CLASS_LABELS[pred_demo[k]]
    conf     = prob_demo[k][pred_demo[k]] * 100
    ok       = pred_demo[k] == true_demo[k]

    color  = 'green' if ok else 'red'
    status = '✓ BENAR' if ok else '✗ SALAH'
    ax.set_title(
        f'{status}\nAktual  : {true_lbl}\nPrediksi: {pred_lbl}\nConf: {conf:.1f}%',
        fontsize=8.5, color=color, fontweight='bold'
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('demo_prediksi_vgg16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: demo_prediksi_vgg16.png')

In [ ]:
# ============================================================
# FUNGSI PREDIKSI GAMBAR BARU (dari file)
# ============================================================

def predict_new_image(image_path):
    """
    Prediksi kelas bunga dari file gambar baru.

    Parameter:
        image_path (str): path ke file gambar

    Contoh penggunaan:
        predict_new_image('foto_bunga.jpg')
    """
    img      = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr      = np.array(img, dtype=np.float32)
    arr_pre  = preprocess_input(arr)
    arr_batch = arr_pre[np.newaxis]

    prob = vgg_e2e.predict(arr_batch, verbose=0)[0]
    pred = np.argmax(prob)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(np.array(img))
    axes[0].axis('off')
    axes[0].set_title('Input Gambar', fontweight='bold')

    bar_c = ['#4CAF50' if i == pred else '#B0BEC5' for i in range(NUM_CLASSES)]
    axes[1].barh(CLASS_LABELS, prob * 100, color=bar_c, edgecolor='black', lw=0.5)
    axes[1].set_xlim([0, 115])
    axes[1].set_xlabel('Probabilitas (%)')
    axes[1].set_title(
        f'VGG16 Fine-Tuning\n→ {CLASS_LABELS[pred]} ({prob[pred]*100:.1f}%)',
        fontweight='bold', color='#2E7D32'
    )
    for i, v in enumerate(prob * 100):
        axes[1].text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9)
    axes[1].grid(True, axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('prediksi_gambar_baru.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Prediksi: {CLASS_LABELS[pred]} (confidence: {prob[pred]*100:.2f}%)')


print('Fungsi predict_new_image() siap.')
print("Contoh: predict_new_image('foto_bunga.jpg')")

In [ ]:
# ============================================================
# SIMPAN MODEL & RINGKASAN AKHIR
# ============================================================

# Simpan model
vgg_e2e.save('model_vgg16_finetuning.keras')
with open('class_names.json', 'w') as f:
    json.dump({'class_names': CLASS_NAMES, 'class_labels': CLASS_LABELS}, f)

print('✅ Model berhasil disimpan:')
print('   • model_vgg16_finetuning.keras')
print('   • class_names.json')

# Simpan metrik ke CSV
pd.DataFrame([vgg_metrics]).to_csv('hasil_vgg16.csv', index=False)
print('   • hasil_vgg16.csv')

# Ringkasan akhir
print('\n' + '=' * 60)
print('  RINGKASAN HASIL — VGG16 FINE-TUNING')
print('=' * 60)
print(f'  Dataset      : {len(X_raw)} gambar | {NUM_CLASSES} kelas')
print(f'  Split        : Train 70% | Val 15% | Test 15%')
print(f'  Augmentasi   : H-Flip, Rotasi, Zoom, Shift, Brightness, Shear')
print(f'  Arsitektur   : VGG16 + GAP + Dense(1024) + Dense(256) + Softmax')
print(f'  Fine-Tuning  : Conv5 block (4 layer terakhir VGG16)')
print(f'  Referensi    : Fei et al. (2023) — IEEE Access 11, 17283')
print()
print(f'  Accuracy     : {vgg_metrics["Accuracy (%)"]:.2f}%')
print(f'  Precision    : {vgg_metrics["Precision"]:.4f}')
print(f'  Recall       : {vgg_metrics["Recall"]:.4f}')
print(f'  Specificity  : {vgg_metrics["Specificity"]:.4f}')
print(f'  F1-Score     : {vgg_metrics["F1-Score"]:.4f}')
print(f'  MCC          : {vgg_metrics["MCC"]:.4f}')
print('=' * 60)
print()
print('  Output Files:')
outputs = [
    'distribusi_dataset.png', 'sampel_dataset.png',
    'data_augmentation.png', 'grafik_training_vgg16.png',
    'confusion_matrix_vgg16.png', 'metrik_per_kelas_vgg16.png',
    'ringkasan_metrik_vgg16.png', 'demo_prediksi_vgg16.png',
    'hasil_vgg16.csv', 'model_vgg16_finetuning.keras'
]
for f in outputs:
    print(f'     • {f}')
print('=' * 60)